# Lab 7 — Tune the Champion, Close the Loop
**SDA-AIE-111 · Day 4 · Hour 5 · 50 minutes · shared with capstone kickoff**

**Objective.** Budgeted randomised search over the full champion pipeline; read the results beyond the winner; execute the final refit + **single** test-set evaluation; deliver the model card.

⏱ The search runs ~10 minutes. Start it, then draft your model card skeleton while it runs — that is deliberate time design, and exactly when you should read the capstone briefs.


In [ ]:
# --- Course setup (identical in every lab) -----------------------------------
import sys, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
RANDOM_STATE = 42

def find_repo_root() -> Path:
    """Locate the course repo root by looking for data/manafeth_customers.parquet."""
    for cand in [Path.cwd(), *Path.cwd().parents]:
        if (cand / "data" / "manafeth_customers.parquet").exists():
            return cand
    raise FileNotFoundError(
        "Course data not found — copy the Manafeth data package files into <repo>/data/")

ROOT = find_repo_root()
DATA = ROOT / "data"
sys.path.insert(0, str(ROOT / "src"))
print("repo root:", ROOT)

In [ ]:
from sklearn.model_selection import train_test_split
from manafeth.features import (build_preprocessor, build_feature_table,
                               MODEL_NUM_COLS, CAT_COLS, LEAKY)
from manafeth.evaluation import CV, evaluate, recall_at_budget, precision_at_budget

customers = pd.read_parquet(DATA / "manafeth_customers.parquet")
orders = pd.read_parquet(DATA / "manafeth_orders.parquet")
full = build_feature_table(customers, orders, pd.Timestamp("2025-11-01"))

X_all = full.drop(columns=["churned_30d", "customer_id", *LEAKY])
y_all = full["churned_30d"]
X_train, X_test, y_train, y_test = train_test_split(
    X_all, y_all, test_size=0.20, stratify=y_all, random_state=RANDOM_STATE)
print("train:", X_train.shape, "| churn:", round(y_train.mean(), 3))

## Task 1 — Pre-tuning checklist *(5 min)*

Fill this in **before** launching the search:

- [ ] Learning-curve verdict (Lab 4): more data won't help → tuning is a legitimate next lever
- [ ] Error-analysis debts (Lab 4 findings) noted — tuning fixes neither of them
- [ ] Leakage tests green: shuffled-label PR-AUC ≈ 0.14 (Lab 3)
- [ ] **Budget statement: 30 draws × 5 folds = 150 fits, decided now, not extended after peeking**


## Task 2 — Launch the search *(15 min, ~10 of them compute)*

In [ ]:
from scipy.stats import loguniform, randint, uniform
from sklearn.model_selection import RandomizedSearchCV
from sklearn.pipeline import Pipeline
from xgboost import XGBClassifier

# TODO: full pipeline (preprocessor + XGBClassifier)
# TODO: search space — log-uniform learning_rate & reg_lambda; integer depth,
#       n_estimators, min_child_weight; uniform subsample/colsample;
#       plus ONE preprocessing knob (imputer strategy)
# TODO: RandomizedSearchCV(n_iter=30, cv=CV, scoring="average_precision",
#       random_state=RANDOM_STATE, refit=True) — launch, then draft the model card


## Task 3 — Read the search beyond the winner *(10 min)*

In [ ]:
# TODO: top-10 table (mean, std, key params) — is the optimum flat or peaked?
# TODO: spearman correlation of each sampled knob with mean_test_score
# TODO: one sentence — what would the NEXT search narrow?


## Task 4 — The final act: the quarantine ends. Once. *(10 min)*

The test partition saved in Lab 1 is opened now, for the first and only time. If the test score sits within ~1 std of the CV estimate, the workflow kept its promise. If the gap exceeds 3 stds — stop and investigate (stale kernel? leak amplified by tuning?) before narrating anything.


In [ ]:
from sklearn.metrics import average_precision_score

# TODO: final_model = search.best_estimator_ (refit already happened)
# TODO: open the quarantined test partition — ONCE. Predict, score PR-AUC.
# TODO: agreement check: |test - CV| in units of CV std; interpret
# TODO: operating point at the 20% budget: recall & precision on test


## Task 5 — The model card *(10 min)*

Complete `project/MODEL_CARD.md` from the template. The card takes the **test** number; `best_score_` is labelled as a selection score. Sections: intended use · data lineage · features + leakage attestation · performance (CV ± std, single test evaluation, operating point) · known weaknesses (your Lab 4 findings) · retraining cadence & owner.

**Commit:** `feat(lab7): tuned champion, test evaluation, model card` — **the golden thread is complete.**

> The sentence to say out loud when the agreement check passes: *the quarantine held, the estimate was honest, the workflow kept its promise.*
